# PhantomMap — Colab quickstart

Repo: https://github.com/ShouyuWang08/PhantomMap

**Before running**: `Runtime -> Change runtime type -> T4 or L4 GPU`. L4 is faster; T4 is free-tier friendly. Every script resumes from its own jsonl output, so it's safe to run across multiple sessions.

**After each split finishes**: download the `results/*.jsonl` file and drop it into your local `~/phantommap/results/` folder.

Workflow:
1. Cells 1–3: setup (once per Colab session).
2. Cell 4 (dry run): 20 samples on Qwen2.5-VL-7B to verify the pipeline before committing to the full runs.
3. Cells 5–6: full runs (long, ~4 h on L4 for all 3 POPE splits × 2 models).
4. Cells 7–8: zip results and peek at headline numbers.
5. Post-processing (atlas + detector + figures + LaTeX) runs LOCALLY on your laptop.

In [ ]:
# 1. Clone (or pull, if you've re-run the notebook after a git push).
import os
if os.path.exists('/content/phantommap/.git'):
    %cd /content/phantommap
    !git pull --ff-only
else:
    !git clone https://github.com/ShouyuWang08/PhantomMap.git /content/phantommap
    %cd /content/phantommap

In [ ]:
# 2. Dependencies. Colab already has torch + CUDA preinstalled.
!pip install -q -U transformers qwen-vl-utils accelerate scikit-learn seaborn scipy tqdm
!python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')"

In [ ]:
# 3. Download data: POPE jsonl (3 splits) + COCO val2014 images referenced by POPE.
#    Uses the official val2014.zip (~6GB download, ~3 min on Colab) for reliability,
#    then extracts just the JPEGs POPE references (~3000 images, ~600 MB on disk).
!python src/download_data.py --out data --coco-mode zip
!echo '--- data dir ---' && ls data/ && echo '--- POPE ---' && ls data/pope/ \
    && echo '--- image count ---' && ls data/coco_val2014/ | wc -l

In [ ]:
# 4. DRY RUN — 20 samples on Qwen2.5-VL-7B-Instruct to verify the pipeline.
#    Takes ~2 min on L4, ~4 min on T4. If this produces 0 records, the script
#    will error out (we fixed the silent-skip bug).
!python src/run_vlm.py --model qwen --split pope_adversarial \
    --out results/dryrun_qwen7b.jsonl --limit 20
!echo '=== line count ===' && wc -l results/dryrun_qwen7b.jsonl
!echo '=== first 3 records ===' && head -n 3 results/dryrun_qwen7b.jsonl

In [ ]:
# 5. FULL RUN — Qwen2.5-VL-7B across all POPE splits.
#    ~80 min / split on T4, ~30 min / split on L4. Resumable.
!python src/run_vlm.py --model qwen --split pope_adversarial --out results/qwen_pope_adversarial.jsonl
!python src/run_vlm.py --model qwen --split pope_popular     --out results/qwen_pope_popular.jsonl
!python src/run_vlm.py --model qwen --split pope_random      --out results/qwen_pope_random.jsonl

In [ ]:
# 6. FULL RUN — LLaVA-NeXT across the same POPE splits.
#    ~80 min / split on T4, ~30 min / split on L4. Resumable.
!python src/run_vlm.py --model llava --split pope_adversarial --out results/llava_pope_adversarial.jsonl
!python src/run_vlm.py --model llava --split pope_popular     --out results/llava_pope_popular.jsonl
!python src/run_vlm.py --model llava --split pope_random      --out results/llava_pope_random.jsonl

In [ ]:
# 7. Zip all jsonls for download. Right-click results_bundle.zip in the file
#    panel on the left, Download, unzip into your local results/ directory.
!cd results && zip -q results_bundle.zip *.jsonl && ls -la results/results_bundle.zip 2>/dev/null || ls -la results_bundle.zip

In [ ]:
# 8. Sanity check: aggregate POPE hallucination rate per split.
import json, glob, sys
sys.path.insert(0, 'src')
from metrics import pope_stats
for p in sorted(glob.glob('results/*_pope_*.jsonl')):
    recs = [json.loads(l) for l in open(p) if l.strip()]
    s = pope_stats(recs)
    print(f'{p}: n={s.n} acc={s.accuracy:.3f} hallu_rate={s.hallucination_rate:.3f} yes_rate={s.yes_rate:.3f}')

## Post-processing — run LOCALLY after downloading `results_bundle.zip`

```bash
# unzip into the local results/ folder
cd ~/phantommap && unzip -o /path/to/results_bundle.zip -d results/

# atlas
python src/atlas.py --inputs results/*.jsonl \
  --out-fig report/figures/fig3_atlas.pdf --out-stats results/atlas_stats.json

# detector per model
python src/detector.py --inputs results/qwen_*.jsonl \
  --model-filter "Qwen/Qwen2.5-VL-7B-Instruct" \
  --out results/detector_qwen.json
python src/detector.py --inputs results/llava_*.jsonl \
  --model-filter "llava-hf/llava-v1.6-mistral-7b-hf" \
  --out results/detector_llava.json

# figures
python src/make_fig2_method.py --out report/figures/fig2_method.pdf
python src/make_figures.py --predictions results/*.jsonl \
  --detector-metrics results/detector_qwen.json results/detector_llava.json

# build PDF
cd report && pdflatex report.tex && bibtex report && pdflatex report.tex && pdflatex report.tex
```